In [ ]:

import pandas as pd
import numpy as np
import gc

from data.dataset import MalwareDatasetLoader
from data.data_processing import split_out_targets, force_dense, preprocess_existing, preprocess_fit

RELOAD_DATA = False
if not RELOAD_DATA:
  try:
    print(df_features_train.head())
  except Exception as e:
    print("No dataframe.  Loading data...")
    RELOAD_DATA=True
if RELOAD_DATA:
  df_loader = MalwareDatasetLoader()

  df_train, df_val, df_test = df_loader.make_data_splits()
  
  df_features_train, df_y_train = split_out_targets(df_train)
  df_features_val, df_y_val = split_out_targets(df_val)
  df_features_test, df_y_test = split_out_targets(df_test)
  del df_loader
  gc.collect()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


def compute_metrics(classifier, df_features, df_y):
  print(f"Compute Metrics Start: {df_features.shape[0]}")
  predictions = classifier.predict(df_features)
  print("Compute Metrics End")

  acc = accuracy_score(df_y, predictions)
  f1 = f1_score(df_y, predictions)
  cm = confusion_matrix(df_y, predictions)

  print(f"Accuracy: {acc:.4f}")
  print(f"F1:       {f1:.4f}")
  #print(f"AUC:      {auc:.4f}")
  print("Confusion matrix:")
  print(cm)
  print()
  return classification_report(df_y, predictions)


In [ ]:
RANDOM_STATE=2025

from data.data_processing import preprocess_fit

import sklearn

def train_bagging(df_features_train, df_y_train, max_depth=2, max_trees=50):
  tree_classifier = sklearn.tree.DecisionTreeClassifier(max_depth=max_depth)

  bagging_classifier = sklearn.ensemble.BaggingClassifier(
    estimator=tree_classifier,
    n_estimators=max_trees,
    max_samples=0.5,
    bootstrap=False,
    n_jobs=16,
    random_state=RANDOM_STATE)

  bagging_classifier.fit(df_features_train, df_y_train)

  return bagging_classifier

max_depth = 10
max_trees = 100
print("Training bagging model")
X_transformed, preprocessor = preprocess_fit(df_features_train, quantile_clipping=True)
bagging_classifier = train_bagging(X_transformed, df_y_train,
                                    max_depth=max_depth,
                                    max_trees=max_trees)
X_validation_transformed = preprocess_existing(df_features_val, preprocessor)

metrics = compute_metrics(bagging_classifier, X_validation_transformed, df_y_val)
print(metrics)

X_test_transformed = preprocess_existing(df_features_test, preprocessor)

metrics_test = compute_metrics(bagging_classifier, X_test_transformed, df_y_test)
print(metrics_test)

In [ ]:
feature_thresholds = {}
# TODO - find unique values, and find the ratio between the deltas between them?

for tree_idx, estimator in enumerate(bagging_classifier.estimators_):
    for node_idx in range(estimator.tree_.node_count):
        feature = estimator.tree_.feature[node_idx]
        if feature not in feature_thresholds:
            feature_thresholds[feature] = []
        feature_thresholds[feature].append(estimator.tree_.threshold[node_idx])
import math
for key, values in feature_thresholds.items():
    sorted_values = sorted(values)
    min_max_ratio = sorted_values[-1] / sorted_values[0]
    if min_max_ratio  > 1.0:
        print(key)
        print(sorted_values)
        sorted_values = [math.log(abs(value) + 2.3) for value in sorted_values]
        print(f"Min-Max Ratio: {min_max_ratio}")
        print(sorted_values)
